# **Import Libary**

In [3]:
import pandas as pd  # Digunakan untuk membuka dan mengelola data dalam bentuk tabel (seperti Excel).
import numpy as np   # Digunakan untuk melakukan hitungan matematika yang rumit dengan cepat.

# import matplotlib.pyplot as plt  # Alat dasar untuk membuat gambar grafik.
# import seaborn as sns             # Alat tambahan agar tampilan grafik lebih cantik dan mudah dibaca.

# Membagi data menjadi dua: satu untuk 'belajar' dan satu lagi untuk 'ujian' (tes).
from sklearn.model_selection import train_test_split

# StandardScaler: Menyamakan skala angka (misal: menyamakan skala antara umur dan gaji).
# OneHotEncoder: Mengubah kata-kata (seperti 'Pria'/'Wanita') menjadi angka agar dimengerti komputer.
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ColumnTransformer & Pipeline: Alat untuk menyusun urutan kerja agar rapi dan otomatis.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Model yang sederhana, cocok untuk tebakan awal (seperti Ya atau Tidak).
from sklearn.linear_model import LogisticRegression

# Model yang bekerja seperti kumpulan pohon keputusan yang saling berdiskusi.
from sklearn.ensemble import RandomForestClassifier

# Model yang sangat cerdas karena belajar dari kesalahan-kesalahan sebelumnya secara cepat.
import xgboost as xgb

# Menampilkan laporan lengkap: berapa banyak tebakan yang benar dan yang salah.
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc

# sns.set_style('darkgrid')           # Memberikan latar belakang kotak-kotak pada grafik agar mudah dibaca.
# plt.rcParams['figure.figsize'] = (12, 7)  # Mengatur ukuran standar gambar grafik (lebar dan tinggi).

# **Memuat & Eksplorasi Data (EDA - Exploratory Data Analysis)**

In [4]:
try:
    # Perintah untuk membaca file bernama 'data2.csv' (seperti membuka file Excel).
    df = pd.read_csv('data2.csv')

    # Jika berhasil, komputer akan memberi tahu kita.
    print("✅ Dataset 'data2.csv' berhasil dimuat.")

except FileNotFoundError:
    # Bagian ini hanya berjalan JIKA file yang dicari tidak ada di sana
    print("❌ ERROR: File 'data2.csv' tidak ditemukan. Pastikan file sudah di-upload ke Colab.")
    exit()

✅ Dataset 'data2.csv' berhasil dimuat.


In [5]:
# Menampilkan 5 baris pertama saja agar layar tidak penuh, tapi kita bisa melihat judul kolom dan contoh datanya.
print("\n5 Baris Pertama Data:")
print(df.head())

# Berapa total barisnya, apa nama kolomnya, dan apakah ada data yang kosong (bolong).
print("\nInformasi Dataset:")
df.info()

# Di sini kita menghitung berapa banyak transaksi yang "Normal" dan berapa banyak yang terdeteksi "Penipuan" (Fraud).
print("\nDistribusi Kelas (Fraud vs Legitimate):")
fraud_counts = df['Fraud'].value_counts()
print(fraud_counts)


5 Baris Pertama Data:
  Profession  Income   Credit_card_number Expiry  Security_code  Fraud
0     DOCTOR   42509     3515418493460774  07/25            251      1
1     DOCTOR   80334      213134223583196  05/32            858      1
2     LAWYER   91552     4869615013764888  03/30            755      1
3     LAWYER   43623      341063356109385  01/29            160      1
4     DOCTOR   22962  4707418777543978402  11/30            102      0

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Profession          10000 non-null  object
 1   Income              10000 non-null  int64 
 2   Credit_card_number  10000 non-null  int64 
 3   Expiry              10000 non-null  object
 4   Security_code       10000 non-null  int64 
 5   Fraud               10000 non-null  int64 
dtypes: int64(4), object(2)
memory usage

# **Pra-pemrosesan Data (Data Preprocessing)**

In [6]:
# Kita membuat daftar kolom yang ingin dibuang.
# Kolom 'Credit Card Number', 'Expiry', dan 'Security Code' adalah data identifier yang sangat unik untuk setiap transaksi.
# Jika kita gunakan model akan 'menghafal' nomor kartu tertentu, bukan belajar pola umum penipuan.
# Ini disebut 'data leakage' dan membuat model tidak berguna untuk data baru.

features_to_drop = ['Credit_card_number', 'Expiry', 'Security_code']

# Perintah untuk menghapus kolom-kolom tersebut dari tabel kita.
df_processed = df.drop(columns=features_to_drop)

# Memberi laporan bahwa kolom sudah berhasil disingkirkan.
print(f"✅ Kolom {features_to_drop} berhasil dibuang.")

✅ Kolom ['Credit_card_number', 'Expiry', 'Security_code'] berhasil dibuang.


In [7]:
# X (Fitur): Kita mengambil semua kolom KECUALI kolom 'Fraud'.
X = df_processed.drop('Fraud', axis=1)

# y (Target): Kita hanya mengambil kolom 'Fraud' saja.
y = df_processed['Fraud']

In [8]:
# Mencari kolom yang isinya angka
# Komputer mengelompokkannya ke dalam daftar 'numerical_features'.
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

# Mencari kolom yang isinya teks atau kategori
# Komputer mengelompokkannya ke dalam daftar 'categorical_features'.
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Menampilkan hasil sortir agar kita bisa memastikan tidak ada data yang salah masuk kategori.
print(f"\nKolom Numerik: {numerical_features}")
print(f"Kolom Kategorikal: {categorical_features}")


Kolom Numerik: ['Income']
Kolom Kategorikal: ['Profession']


In [9]:
# --- Membuat Preprocessing Pipeline ---
# Pipeline ini adalah cara profesional untuk menerapkan transformasi data.
# 1. Untuk data numerik ('Income'): Skalanya akan disamakan menggunakan StandardScaler.
# 2. Untuk data kategorikal ('Profession'): Akan diubah menjadi angka menggunakan OneHotEncoder.
#    'handle_unknown='ignore'' penting agar model tidak error jika menemukan profesi baru di data tes.

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Jaga kolom lain jika ada (untuk fleksibilitas)
)

print("\n✅ Pipeline preprocessing berhasil dibuat.")


✅ Pipeline preprocessing berhasil dibuat.


# **Membagi Data (Train-Test Split)**



In [10]:
# Membagi data menjadi 80% data training dan 20% data testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% data untuk testing
    random_state=42,    # Agar hasil pembagian data selalu sama
    stratify=y          # Menjaga proporsi kelas target
)

print(f"Ukuran Data Training: {X_train.shape}")
print(f"Ukuran Data Testing: {X_test.shape}")

Ukuran Data Training: (8000, 2)
Ukuran Data Testing: (2000, 2)


# **Melatih Model Machine Learning**

In [11]:
# --- Model 1: Logistic Regression (Sebagai Baseline) ---
pipeline_lr = Pipeline(steps=[
    # Pertama: Masukkan data ke "mesin pengolah otomatis" yang sudah kita buat tadi.
    ('preprocessor', preprocessor),

    # Kedua: Gunakan model Logistic Regression
    # 'random_state=42': Agar hasil percobaannya konsisten (tidak berubah-ubah tiap dijalankan).
    # 'class_weight=balanced': Fitur penting! Karena kasus penipuan itu jarang,
    # kita menyuruh model untuk memberi perhatian ekstra pada data penipuan agar tidak terabaikan.
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
])

# PROSES BELAJAR: Komputer mempelajari pola dari data latihan (X_train)
# dan kunci jawabannya (y_train).
pipeline_lr.fit(X_train, y_train)

print("✅ Model Logistic Regression berhasil dilatih.")

✅ Model Logistic Regression berhasil dilatih.


In [12]:
# MODEL 2: RANDOM FOREST (KUMPULAN POHON KEPUTUSAN)

pipeline_rf = Pipeline(steps=[
    # Pertama: Olah data mentah menggunakan mesin yang sama seperti tadi.
    ('preprocessor', preprocessor),

    # Kedua: Gunakan model Random Forest.
    # 'n_estimators=100': Kita membuat 100 "Pohon Keputusan" kecil untuk belajar.
    # 'random_state=42': Supaya "hutan" yang dibuat selalu sama setiap kali kita jalankan.
    # 'class_weight=balanced': Tetap fokus pada kasus penipuan yang jumlahnya sedikit.
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'))
])

pipeline_rf.fit(X_train, y_train)

print("✅ Model Random Forest berhasil dilatih.")

✅ Model Random Forest berhasil dilatih.


In [13]:
# MODEL 3: XGBOOST (MODEL TERCANGGIH)

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

pipeline_xgb = Pipeline(steps=[
    # Pertama: Olah data mentah agar siap pakai.
    ('preprocessor', preprocessor),

    # Kedua: Gunakan model XGBoost.
    # 'scale_pos_weight': Menggunakan hasil hitungan rasio tadi agar model tidak abai pada penipuan.
    # 'eval_metric': Cara komputer mengukur seberapa jauh kesalahannya saat belajar.
    ('classifier', xgb.XGBClassifier(
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight
    ))
])

pipeline_xgb.fit(X_train, y_train)

print("✅ Model XGBoost berhasil dilatih.")

✅ Model XGBoost berhasil dilatih.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:48:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# **Evaluasi Performa Model**

In [14]:

# Membuat daftar (Dictionary) berisi semua model yang sudah kita latih.
models = {
    "Logistic Regression": pipeline_lr,
    "Random Forest": pipeline_rf,
    "XGBoost": pipeline_xgb
}

# Membuat kotak kosong (List) untuk menyimpan skor hasil ujian mereka nanti.
results_summary = []

In [15]:
# Kita memanggil model satu per satu dari grup (looping).
for name, model in models.items():
    print(f"--- Evaluasi Model: {name} ---")

    # y_pred: Jawaban mutlak (Contoh: "Ini Penipuan" atau "Ini Normal").
    # y_prob: Tingkat kepercayaan model (Contoh: "Saya 85% yakin ini Penipuan").
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # 2. LAPORAN HASIL (Classification Report):
    # Menampilkan tabel statistik: seberapa akurat, seberapa teliti,
    # dan seberapa banyak penipuan yang berhasil ditangkap.
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))


    # 4. MENCATAT NILAI RAPOR:
    # Kita mengambil nilai-nilai penting (Precision, Recall, F1, ROC-AUC)
    # untuk disimpan ke dalam daftar peringkat.
    roc_auc = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True)

    results_summary.append({
        'Model': name,
        'Precision (Fraud)': report['1']['precision'], # Ketelitian (agar tidak salah tuduh)
        'Recall (Fraud)': report['1']['recall'],       # Daya tangkap (agar tidak ada penipu lolos)
        'F1-Score (Fraud)': report['1']['f1-score'],   # Keseimbangan antara teliti dan daya tangkap
        'ROC-AUC': roc_auc                             # Skor kepintaran model secara keseluruhan
    })
    print(f"ROC-AUC Score: {roc_auc:.4f}\n")
    print("-" * 60)

--- Evaluasi Model: Logistic Regression ---
Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.52      0.41      0.46       997
       Fraud       0.52      0.63      0.57      1003

    accuracy                           0.52      2000
   macro avg       0.52      0.52      0.51      2000
weighted avg       0.52      0.52      0.51      2000

ROC-AUC Score: 0.5296

------------------------------------------------------------
--- Evaluasi Model: Random Forest ---
Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.50      0.52      0.51       997
       Fraud       0.50      0.48      0.49      1003

    accuracy                           0.50      2000
   macro avg       0.50      0.50      0.50      2000
weighted avg       0.50      0.50      0.50      2000

ROC-AUC Score: 0.5005

------------------------------------------------------------
--- Evaluasi Model: XGBoost ---
Classification Rep

In [16]:
# Menampilkan ringkasan hasil perbandingan model dalam format DataFrame
results_df = pd.DataFrame(results_summary)
print("\nRingkasan Perbandingan Model:")
display(results_df)


Ringkasan Perbandingan Model:


,Model,Precision (Fraud),Recall (Fraud),F1-Score (Fraud),ROC-AUC
0,Logistic Regression,0.518062,0.629113,0.568213,0.529621
1,Random Forest,0.501548,0.484546,0.492901,0.500545
2,XGBoost,0.510574,0.505484,0.508016,0.504278


# HASIL

Logistic Regression (model yang paling simpel) justru menjadi "pemenang" sementara.


**Daya Tangkap (Recall)**: Logistic Regression punya skor 0.62 (62%). Artinya, dari 100 kasus penipuan yang sebenarnya terjadi, dia berhasil menangkap 62 kasus. Random Forest dan XGBoost hanya menangkap sekitar 48-50 kasus.

**Ketelitian (Precision)**: Ketiganya hampir mirip di angka 0.51 (51%). Artinya, jika model menuduh seseorang melakukan penipuan, peluang tuduhan itu benar hanya sekitar 50%. Separuhnya lagi kemungkinan salah tuduh (orang jujur dianggap penipu).


**Mengapa Skornya Terlihat "Rendah"?**

Mungkin kamu bertanya: "Kenapa nilainya cuma 0.5 atau 0.6? Bukannya harusnya 0.9 (90%)?"

Skor di kisaran 0.5 (terutama pada kolom ROC-AUC) menunjukkan bahwa model kamu saat ini hampir sama buruknya dengan menebak koin (untung-untungan).

Skor 0.5: Sama seperti menebak asal-asalan.

Skor 1.0: Sempurna, tidak pernah salah.


 **Masalah Utama: "Ketimpangan Data"**

Melihat hasil ini, ada indikasi kuat bahwa data kamu sangat Imbalanced (transaksi normal jutaan, tapi transaksi penipuan cuma segelintir). Meskipun kita sudah pakai class_weight='balanced', model masih kesulitan membedakan pola penipuan yang nyata dari "kebisingan" data lainnya.

# Aplikasi Model

In [17]:
import joblib

# Kita simpan model terbaik (tadi Logistic Regression yang skornya lumayan)
joblib.dump(pipeline_lr, 'model_fraud.pkl')
print("✅ Model berhasil diunduh!")

✅ Model berhasil diunduh!


# 1. Struktur Folder Proyek
Agar rapi, buatlah susunan folder seperti ini di komputermu:

In [ ]:
proyek-fraud/
│
├── app.py              <-- Kode utama Python (Flask)
├── model_fraud.pkl     <-- Model yang kamu simpan dari Colab
├── templates/          <-- Folder khusus untuk file HTML
│   └── index.html      <-- Tampilan halaman web
└── static/             <-- (Opsional) Untuk file CSS/Gambar

# 2. Kode Backend (app.py)
Kode ini berfungsi sebagai pelayan. Ia menerima data dari user, memberikannya ke model AI, lalu mengirimkan jawabannya kembali ke user.

In [ ]:
# Mengimpor library yang dibutuhkan
from flask import Flask, render_template, request  # Flask untuk web, request untuk ambil input user
import joblib  # Untuk load model machine learning
import pandas as pd  # Untuk mengolah data dalam bentuk tabel (DataFrame)

# Membuat aplikasi Flask
app = Flask(__name__)

# Memuat model yang sudah dilatih (file .pkl)
model = joblib.load('model_fraud.pkl')

# Route untuk halaman utama (homepage)
@app.route('/')
def home():
    # Menampilkan file HTML (index.html) saat pertama kali web dibuka
    return render_template('index.html')

# Route untuk proses prediksi (ketika tombol submit ditekan)
@app.route('/predict', methods=['POST'])
def predict():

    # ================================
    # 1. Mengambil input dari user (form HTML)
    # ================================
    # request.form['nama_input'] harus sama dengan name di HTML
    data = {
        'Income': [float(request.form['income'])],      # Mengambil input income lalu ubah ke float
        'Profession': [request.form['profession']]      # Mengambil input profession (kategori)

        # Jika ada fitur lain (Age, Gender, dll), tambahkan di sini
    }

    # ================================
    # 2. Mengubah data menjadi DataFrame
    # ================================
    # Model ML hanya bisa menerima data dalam bentuk tabel
    df_input = pd.DataFrame(data)

    # ================================
    # 3. Melakukan prediksi menggunakan model
    # ================================
    prediction = model.predict(df_input)  # Output: array, misal [0] atau [1]

    # ================================
    # 4. Mengubah hasil prediksi menjadi teks
    # ================================
    # Jika 1 = Fraud (penipuan), jika 0 = Aman
    hasil = "PENIPUAN" if prediction[0] == 1 else "AMAN"

    # ================================
    # 5. Menampilkan hasil ke halaman web
    # ================================
    return render_template(
        'index.html',
        prediction_text=f'Hasil Deteksi: {hasil}'
    )

# Menjalankan aplikasi
if __name__ == '__main__':
    # debug=True → agar error terlihat saat development
    app.run(debug=True)

# 3. Tampilan Frontend (templates/index.html)
Kita buat formulir sederhana untuk memasukkan data.

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Deteksi Fraud</title>
    <style>
        body { font-family: sans-serif; text-align: center; padding: 50px; }
        form { display: inline-block; text-align: left; border: 1px solid #ccc; padding: 20px; border-radius: 10px; }
        .btn { background: blue; color: white; padding: 10px 20px; border: none; cursor: pointer; }
    </style>
</head>
<body>
    <h1>🛡️ Cek Transaksi Fraud</h1>
    <form action="/predict" method="post">
        <label>Pendapatan (Income):</label><br>
        <input type="number" name="income" required><br><br>

        <label>Pekerjaan (Profession):</label><br>
        <select name="profession">
            <option value="Doctor">Doctor</option>
            <option value="Teacher">Teacher</option>
            <option value="Programmer">Programmer</option>
        </select><br><br>

        <button type="submit" class="btn">Analisis Sekarang</button>
    </form>

    <h2>{{ prediction_text }}</h2>
</body>
</html>

# 4. Install Library
Buka Terminal (di macOS/Linux) atau Command Prompt/PowerShell (di Windows). Install Library yang Dibutuhkan

In [ ]:
pip install flask

In [ ]:
pip install pandas

In [ ]:
pip install joblib

In [ ]:
pip install xgboost

In [ ]:
pip install scikit-learn==1.6.1

Ketik perintah berikut di terminal:

In [ ]:
python app.py

Jika berhasil, kamu akan melihat tulisan seperti ini:
* Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)

Buka browser , lalu ketik alamat http://127.0.0.1:5000 di bar pencarian.

 Halaman web deteksi fraud buatanmu seharusnya sudah muncul.